# Chapter 12 - Dynamic Aperture

Dynamic aperture (DA) is measure of the stable phase space in an accelerator over a specified number of turns. Particles outside of the DA will be driven to large amplitudes where they will be lost. In the following sections, strategies for increasing a ring's DA will be shown in a sample ring.

![Dynamic aperture ray construction](assets/chapter12_dynamic_aperture_rays.svg)

**Figure 1:** The calculation of a dynamic aperture curve in the $x-y$ plane at a given initial $p_z$ value involves calculating aperture curve points (blue dots) along a set of "rays" (dashed lines). The line segments between points is simply for visualization purposes and does not appear in graphs of real data.


In [ ]:
using SciBmad
using CairoMakie
using LinearAlgebra
using Printf


## 12.1 Calculating Dynamic Aperture

Points along rays are tested to determine the dynamic-aperture perimeter. All rays begin at the reference orbit. Along each ray, particles with different starting $(x,y)$ positions are tracked to find the boundary between stable and unstable motion.

For this compact SciBmad example, a particle is classified as stable when:

- every tracked coordinate remains finite;
- the transverse coordinates remain inside a generous $50\ \mathrm{mm}$ numerical guard aperture; and
- the particle completes the requested number of turns.

The guard aperture is not the dynamic aperture itself. It prevents a clearly diverging trajectory from consuming unnecessary tracking time.


In [ ]:
function build_chapter12_ring(;
    n_cells=16,
    chromatic_strength=300.0,
    harmonic_strength=0.0,
    chromatic_sf_strength=chromatic_strength,
    chromatic_sd_strength=-chromatic_strength,
    harmonic_sf_strength=harmonic_strength,
    harmonic_sd_strength=-harmonic_strength,
)
    elements = SciBmad.LineElement[]

    for i in 1:n_cells
        # Odd cells contain the chromatic family; even cells contain the
        # independently adjustable harmonic family introduced in Section 12.2.
        sf = isodd(i) ? chromatic_sf_strength : harmonic_sf_strength
        sd = isodd(i) ? chromatic_sd_strength : harmonic_sd_strength

        append!(elements, [
            Quadrupole(name="QF_$i", L=0.25, Kn1=0.8),
            Drift(name="D1A_$i", L=0.20),
            Sextupole(name="SF_$i", L=0.10, Kn2=sf),
            Drift(name="D1B_$i", L=0.20),
            SBend(name="B_$i", L=1.0, angle=2pi / n_cells),
            Drift(name="D2A_$i", L=0.20),
            Sextupole(name="SD_$i", L=0.10, Kn2=sd),
            Drift(name="D2B_$i", L=0.20),
            Quadrupole(name="QD_$i", L=0.25, Kn1=-0.8),
            Drift(name="D3_$i", L=0.40),
        ])
    end

    return Beamline(elements; species_ref=Species("electron"), E_ref=3e9)
end

ring_chromatic = build_chapter12_ring()
ring_with_harmonic_sextupoles = build_chapter12_ring(
    chromatic_strength=300.0,
    harmonic_strength=300.0,
)

@printf("Tracked elements: %d\n", length(ring_chromatic.line))
@printf("Circumference: %.3f m\n", sum(element.L for element in ring_chromatic.line))


The aperture point on one ray is found in two stages:

1. scan outward in fixed radial steps until the first unstable launch point is found;
2. refine the last stable/first unstable bracket by bisection.

This is a finite-turn numerical definition of DA. Increasing the tracking time can reveal slow losses and therefore reduce the measured aperture.


In [ ]:
function reference_bunch(ring, coordinates)
    coordinates = reshape(coordinates, 1, 6)

    if hasproperty(ring, :p_over_q_ref)
        return Bunch(
            coordinates;
            species=ring.species_ref,
            p_over_q_ref=ring.p_over_q_ref,
        )
    elseif hasproperty(ring, :R_ref)
        return Bunch(
            coordinates;
            species=ring.species_ref,
            R_ref=ring.R_ref,
        )
    else
        return Bunch(coordinates; species=ring.species_ref)
    end
end

function stable_motion(
    ring,
    x,
    y;
    pz=0.0,
    n_turns=80,
    guard_aperture=0.05,
)
    bunch = reference_bunch(ring, [x, 0.0, y, 0.0, 0.0, pz])

    try
        for _ in 1:n_turns
            track!(bunch, ring)
            coordinates = Array(bunch.coords.v)

            all(isfinite, coordinates) || return false
            maximum(abs, coordinates[:, [1, 3]]) < guard_aperture || return false
        end
    catch
        return false
    end

    return true
end

function ray_boundary(
    ring,
    theta;
    pz=0.0,
    n_turns=80,
    r_max=0.03,
    radial_step=0.001,
    refinements=6,
)
    last_stable_radius = 0.0

    for radius in radial_step:radial_step:r_max
        x = radius * cos(theta)
        y = radius * sin(theta)

        if !stable_motion(ring, x, y; pz=pz, n_turns=n_turns)
            low = last_stable_radius
            high = radius

            for _ in 1:refinements
                middle = (low + high) / 2
                middle_is_stable = stable_motion(
                    ring,
                    middle * cos(theta),
                    middle * sin(theta);
                    pz=pz,
                    n_turns=n_turns,
                )
                middle_is_stable ? (low = middle) : (high = middle)
            end

            return low
        end

        last_stable_radius = radius
    end

    return r_max
end

function dynamic_aperture(
    ring;
    pz_values=[0.0],
    n_angles=9,
    n_turns=80,
    r_max=0.03,
)
    angles = range(0, pi; length=n_angles)
    curves = NamedTuple[]

    for pz in pz_values
        radii = [
            ray_boundary(
                ring,
                theta;
                pz=pz,
                n_turns=n_turns,
                r_max=r_max,
            )
            for theta in angles
        ]

        push!(curves, (
            pz=pz,
            x=radii .* cos.(angles),
            y=radii .* sin.(angles),
            radii=radii,
        ))
    end

    return curves
end


The original example evaluates several momentum offsets. For this compact teaching ring, we scan $p_z=0$ through $p_z=0.007$, corresponding to $\delta=0.0\%$ through $0.7\%$ in $0.1\%$ steps. The plot is normalized by nominal beam sizes, $\sigma_x=\sigma_y=1\ \mathrm{mm}$, matching the style of the SciBmad dynamic-aperture example. This demonstration tracks 80 turns and uses 31 rays. Increase `n_turns` and `n_angles` for a production study.


In [ ]:
pz_values = collect(0.000:0.001:0.007)

corrected_da = dynamic_aperture(
    ring_with_harmonic_sextupoles;
    pz_values=pz_values,
    n_angles=31,
    n_turns=80,
    r_max=0.03,
)

for curve in corrected_da
    @printf(
        "pz = %.4f: mean boundary radius = %.2f mm\n",
        curve.pz,
        1e3 * sum(curve.radii) / length(curve.radii),
    )
end

In [ ]:
function plot_dynamic_aperture(
    curves;
    title="Dynamic Aperture",
    sigma_x=1.0e-3,
    sigma_y=1.0e-3,
)
    figure = Figure(size=(820, 600))
    axis = Axis(
        figure[1, 1];
        xlabel="x / sigma_x",
        ylabel="y / sigma_y",
        title=title,
        aspect=DataAspect(),
    )

    x_limit = 0.0
    y_limit = 0.0

    for curve in curves
        x_norm = curve.x ./ sigma_x
        y_norm = curve.y ./ sigma_y
        x_limit = max(x_limit, maximum(abs, x_norm))
        y_limit = max(y_limit, maximum(y_norm))
        label = @sprintf("delta = %.1f%%", 100 * curve.pz)
        lines!(axis, x_norm, y_norm; linewidth=2.8, label=label)
    end

    x_limit = ceil(x_limit)
    y_limit = ceil(y_limit)
    xlims!(axis, -x_limit, x_limit)
    ylims!(axis, 0, y_limit)

    axislegend(axis; position=:rt)
    return figure
end

chromatic_da_figure = plot_dynamic_aperture(
    corrected_da;
    title="Dynamic Aperture with Chromatic and Harmonic Sextupoles",
)
save("assets/chapter12_dynamic_aperture.png", chromatic_da_figure)
chromatic_da_figure


The resulting curves are the SciBmad equivalent of Figure 24 in the original tutorial, plotted in the normalized style used by the SciBmad dynamic-aperture example. Each colored curve represents one momentum offset. The calculated vertices are connected for visualization only, so small jagged features are expected when the scan uses a finite number of rays.

The chromatic-only ring can have a much rougher aperture. Section 12.2 compares it with the corrected ring to show why harmonic sextupoles are useful.


## 12.2 Example: Adding Sextupoles

A common element used for DA optimization are sextupoles. "Harmonic" sextupoles, that is, sextupoles that minimize resonance driving modes, can be used to increase the transverse aperture. Sextupoles can be added to the lattice by following the steps below.

The original Bmad example performs the following structural changes:

1. define chromatic sextupoles in dispersive arc regions;
2. define harmonic sextupoles in straight sections;
3. split drifts to make room for the new elements;
4. modify the forward and reverse FODO cells;
5. rebuild the ring from the altered cells.

Our compact ring already splits the drifts around every sextupole location. Odd cells contain the chromatic families `SF` and `SD`. In the first calculation the even-cell harmonic families were present but had zero strength. We now turn those families on without changing the linear quadrupole lattice.


In [ ]:
ring_with_harmonic_sextupoles = build_chapter12_ring(
    chromatic_strength=300.0,
    harmonic_strength=300.0,
)

@printf("Chromatic SF/SD strength magnitude: %.1f\n", 300.0)
@printf("Harmonic  SF/SD strength magnitude: %.1f\n", 300.0)


To see whether the added harmonic family helps, we repeat the on-momentum ray search. This is the essential DA optimization loop: change sextupole settings, recalculate the aperture, and retain changes that improve the desired stability measure.


In [ ]:
chromatic_only_da = dynamic_aperture(
    ring_chromatic;
    pz_values=[0.0],
    n_angles=31,
    n_turns=80,
    r_max=0.03,
)

harmonic_da = dynamic_aperture(
    ring_with_harmonic_sextupoles;
    pz_values=[0.0],
    n_angles=31,
    n_turns=80,
    r_max=0.03,
)

before_mean = 1e3 * sum(chromatic_only_da[1].radii) / length(chromatic_only_da[1].radii)
after_mean = 1e3 * sum(harmonic_da[1].radii) / length(harmonic_da[1].radii)

@printf("Mean on-momentum boundary before harmonic sextupoles: %.2f mm\n", before_mean)
@printf("Mean on-momentum boundary after harmonic sextupoles:  %.2f mm\n", after_mean)

In [ ]:
comparison_figure = Figure(size=(820, 600))
comparison_axis = Axis(
    comparison_figure[1, 1];
    xlabel="x / sigma_x",
    ylabel="y / sigma_y",
    title="Effect of Adding Harmonic Sextupoles",
    aspect=DataAspect(),
)

sigma_x = 1.0e-3
sigma_y = 1.0e-3

lines!(
    comparison_axis,
    chromatic_only_da[1].x ./ sigma_x,
    chromatic_only_da[1].y ./ sigma_y;
    linewidth=2.8,
    label="chromatic families only",
)

lines!(
    comparison_axis,
    harmonic_da[1].x ./ sigma_x,
    harmonic_da[1].y ./ sigma_y;
    linewidth=2.8,
    label="chromatic + harmonic families",
)

comparison_x_limit = ceil(max(
    maximum(abs, chromatic_only_da[1].x ./ sigma_x),
    maximum(abs, harmonic_da[1].x ./ sigma_x),
))
comparison_y_limit = ceil(max(
    maximum(chromatic_only_da[1].y ./ sigma_y),
    maximum(harmonic_da[1].y ./ sigma_y),
))
xlims!(comparison_axis, -comparison_x_limit, comparison_x_limit)
ylims!(comparison_axis, 0, comparison_y_limit)

axislegend(comparison_axis; position=:rt)
save("assets/chapter12_adding_sextupoles.png", comparison_figure)
comparison_figure


For this demonstration setting, the harmonic family increases the mean on-momentum boundary radius. This result is not a general rule that stronger sextupoles always improve DA. Sextupole strengths and phase relationships must be optimized, and the final settings must be checked over the required momentum range and tracking time.


## 12.3 Example: Chromaticity Correction

The Bmad lattice for Section 12.3 introduces two overlay knobs for the arc chromatic sextupoles:

```bmad
OSF: overlay = {SF%_*[k2]: x}, var = {x}
OSD: overlay = {SD%_*[k2]: x}, var = {x}
```

The matching `tao.init` file varies those two knobs and sets the targets `chrom.a = 1` and `chrom.b = 1`. In this SciBmad notebook, we keep the same idea: one knob changes all chromatic `SF` sextupoles, and the other changes all chromatic `SD` sextupoles. The full Bmad ESR lattice is not rebuilt here; instead, the same two-family correction is applied to the compact teaching ring used above.

With Julia 1.12 and the current SciBmad API, `twiss` does not take a `pz` keyword, and the transverse tune TPS coefficients returned for this coasting compact ring do not directly give the off-momentum chromaticity. Therefore chromaticity is measured by tracking small transverse perturbations for one turn, constructing the transverse one-turn matrix at `+dpz` and `-dpz`, and differencing the resulting tunes.


In [ ]:
function ring_with_os_knobs(os; harmonic_strength=0.0)
    return build_chapter12_ring(
        chromatic_sf_strength=os[1],
        chromatic_sd_strength=os[2],
        harmonic_strength=harmonic_strength,
    )
end

function one_turn_transverse_coordinates(ring, u; pz=0.0)
    bunch = reference_bunch(ring, [u[1], u[2], u[3], u[4], 0.0, pz])
    track!(bunch, ring)
    coordinates = Array(bunch.coords.v)
    return vec(coordinates[1, [1, 2, 3, 4]])
end

function one_turn_transverse_matrix(ring; pz=0.0, step=1e-7)
    matrix = zeros(4, 4)
    origin = zeros(4)

    for j in 1:4
        perturbation = zeros(4)
        perturbation[j] = step
        matrix[:, j] = (
            one_turn_transverse_coordinates(ring, origin .+ perturbation; pz=pz) .-
            one_turn_transverse_coordinates(ring, origin .- perturbation; pz=pz)
        ) ./ (2step)
    end

    return matrix
end

function tunes_from_matrix(matrix)
    eigenvalues = eigvals(matrix)
    tunes = sort(mod.(angle.(eigenvalues[imag.(eigenvalues) .> 0]) ./ (2pi), 1.0))
    length(tunes) == 2 || error("Expected two stable transverse modes.")
    return tunes
end

function tunes_from_tracking(ring; pz=0.0)
    return tunes_from_matrix(one_turn_transverse_matrix(ring; pz=pz))
end

function chromaticity_from_tracking(os; harmonic_strength=0.0, dpz=1e-4)
    ring = ring_with_os_knobs(os; harmonic_strength=harmonic_strength)
    return (
        tunes_from_tracking(ring; pz=dpz) .-
        tunes_from_tracking(ring; pz=-dpz)
    ) ./ (2dpz)
end

function chromaticity_response(os; harmonic_strength=0.0, knob_step=0.01)
    response = zeros(2, 2)

    for j in 1:2
        step_vector = zeros(2)
        step_vector[j] = knob_step
        response[:, j] = (
            chromaticity_from_tracking(os .+ step_vector; harmonic_strength=harmonic_strength) .-
            chromaticity_from_tracking(os .- step_vector; harmonic_strength=harmonic_strength)
        ) ./ (2knob_step)
    end

    return response
end

function correct_chromaticity(os0; target=[1.0, 1.0], max_iter=5, tolerance=1e-5)
    os = copy(os0)
    history = NamedTuple[]

    for iteration in 1:max_iter
        chromaticity = chromaticity_from_tracking(os)
        residual = target .- chromaticity
        push!(history, (iteration=iteration, os=copy(os), chromaticity=chromaticity))
        norm(residual) < tolerance && break

        response = chromaticity_response(os)
        os .+= response \ residual
    end

    return os, history
end

initial_os = [2.0, -2.0]
target_chromaticity = [1.0, 1.0]
corrected_os, correction_history = correct_chromaticity(initial_os; target=target_chromaticity)
corrected_chromaticity = chromaticity_from_tracking(corrected_os)
ring_chromaticity_corrected = ring_with_os_knobs(corrected_os)

println("Chromaticity correction history:")
for step in correction_history
    @printf(
        "  iter %d: OSF = % .6f, OSD = % .6f, chrom = (% .6f, % .6f)\n",
        step.iteration,
        step.os[1],
        step.os[2],
        step.chromaticity[1],
        step.chromaticity[2],
    )
end

@printf("\nCorrected OSF = %.6f\n", corrected_os[1])
@printf("Corrected OSD = %.6f\n", corrected_os[2])
@printf("Corrected chromaticities = (%.6f, %.6f)\n", corrected_chromaticity[1], corrected_chromaticity[2])
